# Filled posts experiments: compare with baseline (ticket 2166)

One table per experiment: baseline against variant for the three rule metrics, per fold, with a keep or reject decision.

In [ ]:
import io
import json

import boto3
import polars as pl

RESULTS_PREFIX = "s3://skillsforcare/emr-notebooks/filled-posts-eval/"
SUFFIX = ""  # "" for the main run, "_seed43" for the second-seed rescore of winners
BASELINE = f"baseline{SUFFIX}"

SLOPE_TOLERANCE = 0.005  # per year, on the absolute slope
JUMPINESS_TOLERANCE = 0.05  # relative
VARIANTS = [
 [
  "time_months_ch",
  "care_home_model",
  "months elapsed replaces the date index"
 ],
 [
  "time_months_nr",
  "non_res_with_dormancy_model",
  "months elapsed replaces the date index and its cube"
 ],
 [
  "time_months_nr_nocubic",
  "non_res_with_dormancy_model",
  "months elapsed, no cubic term"
 ],
 [
  "time_index_nr_nocubic",
  "non_res_with_dormancy_model",
  "baseline date index, no cubic term (control)"
 ],
 [
  "dormant_cap24_nr",
  "non_res_with_dormancy_model",
  "ever_dormant + months since dormant capped at 24"
 ],
 [
  "dormant_cap36_nr",
  "non_res_with_dormancy_model",
  "ever_dormant + months since dormant capped at 36"
 ],
 [
  "dormant_cap48_nr",
  "non_res_with_dormancy_model",
  "ever_dormant + months since dormant capped at 48"
 ],
 [
  "time_registered_cap120_nr",
  "non_res_with_dormancy_model",
  "time registered capped at 120 months"
 ],
 [
  "maxiter10000_nr",
  "non_res_with_dormancy_model",
  "alpha 0.001 refit with max_iter=10000"
 ],
 [
  "alpha_0p0001_ch",
  "care_home_model",
  "alpha 0.0001"
 ],
 [
  "alpha_0p001_ch",
  "care_home_model",
  "alpha 0.001"
 ],
 [
  "alpha_0p01_ch",
  "care_home_model",
  "alpha 0.01"
 ],
 [
  "alpha_0p1_ch",
  "care_home_model",
  "alpha 0.1"
 ],
 [
  "alpha_0p0001_nr",
  "non_res_with_dormancy_model",
  "alpha 0.0001"
 ],
 [
  "alpha_0p001_nr",
  "non_res_with_dormancy_model",
  "alpha 0.001"
 ],
 [
  "alpha_0p01_nr",
  "non_res_with_dormancy_model",
  "alpha 0.01"
 ],
 [
  "alpha_0p1_nr",
  "non_res_with_dormancy_model",
  "alpha 0.1"
 ]
]

bucket, prefix = RESULTS_PREFIX.removeprefix("s3://").split("/", 1)
s3 = boto3.client("s3")


def load(run_label):
    body = s3.get_object(Bucket=bucket, Key=f"{prefix}{run_label}/results.parquet")["Body"].read()
    return pl.read_parquet(io.BytesIO(body))


def metric(df, model, level, group, name):
    """fold -> value for one metric."""
    rows = df.filter(
        (pl.col("model") == model) & (pl.col("level") == level) & (pl.col("group") == group) & (pl.col("metric") == name)
    )
    return dict(zip(rows["fold"], rows["value"]))

In [ ]:
def decide(base_df, var_df, model):
    err = "weighted_absolute_percentage_error"
    err_b = metric(base_df, model, "headline_group_totals", "all", err)
    err_v = metric(var_df, model, "headline_group_totals", "all", err)
    r2_b = metric(base_df, model, "headline_group_totals", "all", "r2")
    r2_v = metric(var_df, model, "headline_group_totals", "all", "r2")
    folds = sorted(f for f in err_b if f != "pooled")
    wins = [err_v[f] < err_b[f] for f in folds]
    headline_ok = all(wins)

    slope_groups = (
        base_df.filter(
            (pl.col("model") == model) & (pl.col("level") == "imputed_target_period_bias") & (pl.col("metric") == "bias_slope_per_year")
        )["group"].unique().sort().to_list()
    )
    slopes, slope_ok = [], True
    for g in slope_groups:
        b = metric(base_df, model, "imputed_target_period_bias", g, "bias_slope_per_year")["pooled"]
        v = metric(var_df, model, "imputed_target_period_bias", g, "bias_slope_per_year")["pooled"]
        ok = abs(v) <= abs(b) + SLOPE_TOLERANCE
        slope_ok &= ok
        slopes.append((g, b, v, ok))

    jb = metric(base_df, model, "jumpiness", "all", "mean_period_to_period_change")["pooled"]
    jv = metric(var_df, model, "jumpiness", "all", "mean_period_to_period_change")["pooled"]
    jump_ok = jv <= jb * (1 + JUMPINESS_TOLERANCE)

    lines = [f"**{model}**", "", "| fold | baseline error | variant error | beats | baseline R2 | variant R2 |", "|---|---|---|---|---|---|"]
    for f, w in zip(folds, wins):
        lines.append(f"| {f} | {err_b[f]:.2%} | {err_v[f]:.2%} | {'yes' if w else 'no'} | {r2_b[f]:.3f} | {r2_v[f]:.3f} |")
    lines.append(f"| pooled | {err_b['pooled']:.2%} | {err_v['pooled']:.2%} | | {r2_b['pooled']:.3f} | {r2_v['pooled']:.3f} |")
    lines += ["", "| trend (imputed target) | baseline slope/yr | variant slope/yr | no worse |", "|---|---|---|---|"]
    lines += [f"| {g} | {b:+.4f} | {v:+.4f} | {'yes' if ok else 'no'} |" for g, b, v, ok in slopes]
    lines += ["", f"Jumpiness: baseline {jb:.3%}, variant {jv:.3%} ({jv / jb - 1:+.1%}), {'ok' if jump_ok else 'too jumpy'}", ""]
    keep = headline_ok and slope_ok and jump_ok
    verdict = "KEEP" if keep else "REJECT"
    lines.append(
        f"Headline {'pass' if headline_ok else 'fail'} ({sum(wins)}/{len(folds)} folds), "
        f"trend {'pass' if slope_ok else 'fail'}, jumpiness {'pass' if jump_ok else 'fail'} -> **{verdict}**"
    )
    return "\n".join(lines), keep

In [ ]:
base_df = load(BASELINE)

# Self-check: the baseline compared with itself never beats itself.
_, kept = decide(base_df, base_df, "care_home_model")
assert not kept, "decision logic broken"

verdicts = {}
for variant, model, description in VARIANTS:
    try:
        var_df = load(f"{variant}{SUFFIX}")
    except s3.exceptions.NoSuchKey:
        continue
    text, keep = decide(base_df, var_df, model)
    verdicts[variant] = keep
    print(f"### {variant}: {description}\n\n{text}\n")
print({k: ("KEEP" if v else "reject") for k, v in verdicts.items()})

## Penalty: convergence and zero coefficients

`hit_max_iter` = 1 means the fit stopped at `max_iter`. Zero coefficients are from the all-rows fit.

In [ ]:
pl.Config.set_tbl_rows(100)
rows = []
for variant, model, _ in VARIANTS:
    if not (variant.startswith("alpha_") or variant.startswith("maxiter")):
        continue
    try:
        df = load(f"{variant}{SUFFIX}")
    except s3.exceptions.NoSuchKey:
        continue
    d = df.filter((pl.col("model") == model) & (pl.col("level") == "fit_diagnostics")).pivot(on="metric", index="fold", values="value")
    rows.append(d.with_columns(pl.lit(variant).alias("variant")))
if rows:
    print(pl.concat(rows, how="diagonal").select("variant", "fold", "n_iter", "max_iter", "hit_max_iter", "zero_coefficients", "number_of_features"))

for variant, model, _ in VARIANTS:
    if variant.startswith("alpha_"):
        try:
            body = s3.get_object(Bucket=bucket, Key=f"{prefix}{variant}{SUFFIX}/fit_diagnostics.json")["Body"].read()
        except s3.exceptions.NoSuchKey:
            continue
        zero = [r["zero_features"] for r in json.loads(body) if r["model"] == model and r["fit"] == "all_rows"][0]
        print(variant, "exactly-zero coefficients:", zero or "none")

## Reproduction check

`alpha_0p001_*` is the baseline spec re-run. Its scores should equal the baseline's; if not, the modelling code or data has changed since the baseline.

In [ ]:
for variant, model in (("alpha_0p001_ch", "care_home_model"), ("alpha_0p001_nr", "non_res_with_dormancy_model")):
    try:
        var_df = load(f"{variant}{SUFFIX}")
    except s3.exceptions.NoSuchKey:
        continue
    b = metric(base_df, model, "headline_group_totals", "all", "weighted_absolute_percentage_error")
    v = metric(var_df, model, "headline_group_totals", "all", "weighted_absolute_percentage_error")
    worst = max(abs(b[f] - v[f]) for f in b)
    print(variant, "max difference in headline error:", f"{worst:.2e}", "(reproduces)" if worst < 1e-6 else "(DIFFERS)")